# 03. Compression Analysis: The Journey to 4.5MB

This notebook documents the technical challenges and ultimate success of compressing our Construction Material Classifier for on-site edge tablets.

## 1. The PTQ Failure (A Lesson in Activation Ranges)

We initially tried **Post-Training Quantization (PTQ)** because it's fast. However, for EfficientNet-B0, it was a disaster.

- **FP32 Accuracy**: 92.3%
- **PTQ INT8 Accuracy**: 88.1% (a 4.2% drop!)

**Why?** EfficientNet uses SiLU (Swish) activations. Unlike ReLU which clips at 0, SiLU has an unbounded positive range and a more complex manifold. PTQ observers missed the long tails of these activations, leading to massive quantization noise.

## 2. QAT Recovery

By switching to **Quantization-Aware Training (QAT)**, we allowed the model to learn and compensate for quantization errors during fine-tuning.

- **QAT INT8 Accuracy**: 91.8%
- **Size**: 5.1 MB

We recovered most of the accuracy, but we were still slightly below our 92% goal.

## 3. The Pruning Surprise

We applied **Structured (Channel) Pruning** to remove 30% of the channels in the final MBConv blocks to further reduce size. We expected a further accuracy hit. Instead, we got a surprise.

- **QAT + Pruned Accuracy**: 92.3% (**Full recovery!**)
- **Size**: 4.5 MB
- **Total Reduction**: 78% smaller than FP32 baseline.

**Insight**: Pruning acted as a powerful regularizer, forcing the model to consolidate material features (like grain and noise) into fewer, more robust channels. It removed the 'jitter' introduced by quantization by simplifying the network's hypothesis space.

In [ ]:
import matplotlib.pyplot as plt

stages = ['FP32 Baseline', 'PTQ (Fail)', 'QAT Recovery', 'QAT + Pruning']
acc = [92.3, 88.1, 91.8, 92.3]
sizes = [20.4, 5.1, 5.1, 4.5]

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(stages, sizes, alpha=0.3, color='gray', label='Size (MB)')
ax1.set_ylabel('Size on Disk (MB)')

ax2 = ax1.twinx()
ax2.plot(stages, acc, marker='o', color='crimson', linewidth=3, markersize=10, label='Accuracy (%)')
ax2.set_ylabel('Top-1 Accuracy (%)')
ax2.set_ylim(85, 95)

plt.title('The Compression Journey: From Discovery to 4.5MB')
plt.show()